In [10]:
import langchain 

In [11]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [12]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### Simple LLM call with streaming 


In [13]:
# Simple LLM call with streaming 
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage,SystemMessage 

In [14]:
model = init_chat_model("groq:openai/gpt-oss-20b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000020A825F6510>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020A825F7230>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [15]:
# create messages
messages = [
    SystemMessage("You are a helpful AI assistant"),
    HumanMessage("what are the top 2 benefits of using Langchain?")
]

In [16]:
#invoke tha model
response = model.invoke(messages)
response

AIMessage(content='**Top 2 Benefits of Using LangChain**\n\n| # | Benefit | Why It Matters |\n|---|---------|----------------|\n| 1 | **Modular, Plug‑and‑Play Architecture** | LangChain’s chain, memory, and tool modules let you mix and match components (LLMs, APIs, databases, etc.) without rewriting boilerplate. This speeds up prototyping, reduces code duplication, and makes your app easy to evolve. |\n| 2 | **Advanced Prompt & Context Management** | Built‑in tools for prompt templating, dynamic prompt construction, and stateful memory (short‑term & long‑term) give you fine‑grained control over how the model interprets context. It enables more reliable, consistent, and conversational AI behavior across sessions. |\n\nThese two strengths—rapid, reusable architecture and sophisticated prompt/context handling—are what most users cite as the biggest gains when adopting LangChain.', additional_kwargs={'reasoning_content': 'The user asks: "what are the top 2 benefits of using Langchain?" The

In [17]:
print(response.content)

**Top 2 Benefits of Using LangChain**

| # | Benefit | Why It Matters |
|---|---------|----------------|
| 1 | **Modular, Plug‑and‑Play Architecture** | LangChain’s chain, memory, and tool modules let you mix and match components (LLMs, APIs, databases, etc.) without rewriting boilerplate. This speeds up prototyping, reduces code duplication, and makes your app easy to evolve. |
| 2 | **Advanced Prompt & Context Management** | Built‑in tools for prompt templating, dynamic prompt construction, and stateful memory (short‑term & long‑term) give you fine‑grained control over how the model interprets context. It enables more reliable, consistent, and conversational AI behavior across sessions. |

These two strengths—rapid, reusable architecture and sophisticated prompt/context handling—are what most users cite as the biggest gains when adopting LangChain.


In [18]:
model.invoke([HumanMessage("what is machine learniing?")])

AIMessage(content='**Machine Learning (ML)** is a branch of computer science that teaches computers how to learn patterns and make decisions from data, without being explicitly programmed for each task.\n\n---\n\n## The Core Idea\n1. **Data** – The raw material: numbers, images, text, sensor readings, etc.\n2. **Model** – A mathematical representation (e.g., a neural network, decision tree, or statistical equation) that can be adjusted.\n3. **Learning Algorithm** – A procedure that tweaks the model’s parameters so the model’s predictions or decisions become more accurate on the data.\n4. **Evaluation** – Measuring how well the model performs, usually on a separate set of data it hasn’t seen before.\n\n---\n\n## How It Works (Simplified)\n\n| Step | What Happens | Example |\n|------|--------------|---------|\n| **Collect data** | Gather examples of the problem you want to solve. | Thousands of photos of cats and dogs. |\n| **Choose a model** | Pick a structure that can represent the rel

In [19]:
## Streaming Example
for chunk in model.stream(messages):
    print(chunk.content, end="", flush=True)


**Top 2 Benefits of Using LangChain**

| # | Benefit | Why It Matters |
|---|---------|----------------|
| **1. Rapid, Modular Development** | LangChain gives you a library of pre‑built, interchangeable components—retrievers, prompt templates, memory stores, chain builders, etc. You can drop‑in the right piece, wire it up with a few lines of code, and get a working LLM application (chatbot, summarizer, QA system, etc.) in minutes instead of weeks. | • Cuts down on boilerplate and repetitive coding.<br>• Lets you iterate faster by swapping out parts (e.g., try a different embedding model or memory type). |
| **2. Seamless, Provider‑agnostic Integration** | LangChain abstracts over dozens of LLM APIs (OpenAI, Anthropic, Cohere, Gemini, Azure, etc.) and downstream tools (search engines, databases, APIs). You write a chain once and it can run on any supported backend with minimal changes. | • Future‑proof your code against vendor lock‑in.<br>• Easily switch or combine models to optimize fo

## Dynamic prompt Templates


In [20]:
from langchain_core.prompts import ChatPromptTemplate

## create translation app

translation_template = ChatPromptTemplate.from_messages([
    ("system", "You are a professional translator. Translate the follow text {text} from {source_language} to {target_language}. Maintain the tone and style"),
    ("user","{text}")
])

## using the template
prompt = translation_template.invoke({
    "source_language":"English",
    "target_language":"Spanish",
    "text":"Langchain makes building AI application incredibly easy!"
})

In [21]:
prompt

ChatPromptValue(messages=[SystemMessage(content='You are a professional translator. Translate the follow text Langchain makes building AI application incredibly easy! from English to Spanish. Maintain the tone and style', additional_kwargs={}, response_metadata={}), HumanMessage(content='Langchain makes building AI application incredibly easy!', additional_kwargs={}, response_metadata={})])

In [22]:
translated_response = model.invoke(prompt)
print(translated_response.content)

¡Langchain hace que construir aplicaciones de IA sea increíblemente fácil!


## Building your first chain

In [27]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Create a more complex chain
def create_story_chain():
    # Template for story generation
    story_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a creative storyteller. Write a short, engaging story based on the given theme."),
        ("user", "Theme: {theme}\nMain character: {character}\nSetting: {setting}")
    ])
    
    # Template for story analysis
    analysis_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a literary critic. Analyze the following story and provide insights."),
        ("user", "{story}")
    ])
    
    # Build the chain - Method 1: Sequential execution
    story_chain = (
        story_prompt 
        | model 
        | StrOutputParser()
    )
    
    # Create a function to pass the story to analysis
    def analyze_story(story_text):
        return {"story": story_text}
    
    analysis_chain = (
        story_chain
        | RunnableLambda(analyze_story)
        | analysis_prompt
        | model
        | StrOutputParser()
    )
    return analysis_chain

In [31]:
chain = create_story_chain()
chain

ChatPromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a creative storyteller. Write a short, engaging story based on the given theme.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, template='Theme: {theme}\nMain character: {character}\nSetting: {setting}'), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': 

In [32]:
result = chain.invoke({
    "theme": "artificial intelligence",
    "character": "a curious robot",
    "setting": "a futuristic city"
})

print("Story and Analysis:")
print(result)

Story and Analysis:
**The Neon Library – A Literary Critique**

| Element | What the story does well | What could be sharpened |
|---------|--------------------------|------------------------|
| **Setting** | The city of Neo‑Tokyo is rendered with a sensory richness that feels both hyper‑real and intimate. The “sky‑bridges,” “floating gardens,” and the “glowing holograms that danced like fireflies” immediately anchor the reader in a world that is at once futuristic and eerily familiar. | The visual imagery can at times feel repetitive (“glowing holograms…fireflies”), and a few more concrete details—such as the specific sounds of autonomous vehicles or the texture of the transit hub’s concrete—could ground the reader more firmly. |
| **Tone & Mood** | The narrative voice is contemplative and almost lyrical, matching the theme of curiosity. The gradual shift from routine to wonder is mirrored in the pacing, which slows as B‑7 ventures into the library. | The lyrical quality occasionally 